# Multi-Design Probe Comparison

Applies three probe methods to hidden states extracted with each of the six extraction designs from notebook 04b.

**Probe methods**
| Method | Description |
|--------|-------------|
| **LR** | Logistic Regression, `StandardScaler + C=0.01`, lbfgs |
| **LAT** | Linear Activation Trajectory reading vector — mean-difference direction, normalized |
| **MLP** | `StandardScaler → PCA(50) → MLP(64, relu)`, early stopping |

**Extraction designs**
| Design | Method | x-axis |
|--------|--------|--------|
| **A** | Last token of each assistant response (causal, one fwd pass) | turn_idx |
| **B** | Last token of final assistant response, varying k prior turns in context | k (context depth) |
| **C** | Last token of each user message | turn_idx |
| **D** | First token of each assistant response | turn_idx |
| **E** | Mean pool across all assistant response tokens | turn_idx |
| **F** | Final turn only, prior assistant responses masked as `.` | single point |

**Probe tasks**
- **Task 1 — Harmful vs Benign**: predict `label` (1 = harmful, 0 = benign). Trains on pairs 0–79, tests on 80–99.
- **Task 2 — Outcome probe**: predict `jailbroken vs refusal` *within harmful conversations only*. Same train/test split.

Run this notebook after running 04b for both harmful and benign folders.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from IPython.display import display

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── Config ────────────────────────────────────────────────────────────────────
FRAMEWORK = "actorattack"   # ← change to process a different framework

HARMFUL_FOLDER = "actorattack_harmful_v2"   # ← harmful representations folder (base name)
BENIGN_FOLDER  = "actorattack_benign_v3"    # ← benign  representations folder (base name)

DESIGNS = ["A", "B", "C", "D", "E", "F"]   # designs to evaluate

REPR_ROOT = repo_root / "data" / "representations"

TRAIN_MAX_PAIR_ID = 79   # pairs  0–79  → train
TEST_MIN_PAIR_ID  = 80   # pairs 80–99  → test
MIN_EXAMPLES      = 30   # min per class per turn to train a probe
MIN_TEST          = 5    # min per class in test set

# MLP hyperparameters
MLP_N_COMPONENTS = 50
MLP_HIDDEN       = 64
MLP_ALPHA        = 0.1

FIG_DIR = repo_root / "notebooks" / "figures" / FRAMEWORK
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Framework: {FRAMEWORK}")
print(f"Harmful folder: {HARMFUL_FOLDER}")
print(f"Benign folder:  {BENIGN_FOLDER}")
print(f"Designs: {DESIGNS}")
print(f"Figures → {FIG_DIR}")

In [ ]:
# ── Probe helper functions ────────────────────────────────────────────────────

def probe_lr(X_tr, y_tr, X_te, y_te):
    """Logistic regression with StandardScaler and strong L2 regularization."""
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    clf = LogisticRegression(max_iter=5000, C=0.01, random_state=42)
    clf.fit(X_tr_s, y_tr)
    probs = clf.predict_proba(X_te_s)[:, 1]
    return {
        "auc": roc_auc_score(y_te, probs),
        "acc": accuracy_score(y_te, clf.predict(X_te_s)),
    }


def probe_lat(X_tr, y_tr, X_te, y_te):
    """LAT reading vector: normalized mean-difference direction."""
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr.astype(np.float32))
    X_te_s = scaler.transform(X_te.astype(np.float32))
    mu_pos = X_tr_s[y_tr == 1].mean(axis=0)
    mu_neg = X_tr_s[y_tr == 0].mean(axis=0)
    v = mu_pos - mu_neg
    v = v / (np.linalg.norm(v) + 1e-12)
    scores = X_te_s @ v
    return {
        "auc": roc_auc_score(y_te, scores),
        "acc": accuracy_score(y_te, (scores > 0).astype(int)),
    }


def probe_mlp(X_tr, y_tr, X_te, y_te):
    """MLP probe: StandardScaler → PCA(50) → MLP(64, relu), early stopping."""
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("pca",    PCA(n_components=MLP_N_COMPONENTS, random_state=42)),
        ("mlp",    MLPClassifier(
            hidden_layer_sizes=(MLP_HIDDEN,),
            activation="relu",
            alpha=MLP_ALPHA,
            max_iter=500,
            random_state=42,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=20,
        )),
    ])
    pipe.fit(X_tr, y_tr)
    probs = pipe.predict_proba(X_te)[:, 1]
    return {
        "auc": roc_auc_score(y_te, probs),
        "acc": accuracy_score(y_te, pipe.predict(X_te)),
    }


def probe_all(X_tr, y_tr, X_te, y_te):
    """Run all three methods and return a combined dict."""
    lr  = probe_lr(X_tr, y_tr, X_te, y_te)
    lat = probe_lat(X_tr, y_tr, X_te, y_te)
    mlp = probe_mlp(X_tr, y_tr, X_te, y_te)
    return {
        "lr_auc":  lr["auc"],  "lr_acc":  lr["acc"],
        "lat_auc": lat["auc"], "lat_acc": lat["acc"],
        "mlp_auc": mlp["auc"], "mlp_acc": mlp["acc"],
        "n_train": len(y_tr), "n_test": len(y_te),
        "n_tr_pos": (y_tr == 1).sum(), "n_tr_neg": (y_tr == 0).sum(),
    }


print("Probe helpers defined.")

In [ ]:
# ── Data loading helpers ──────────────────────────────────────────────────────

def load_design(base_folder, design_letter):
    """
    Load hidden states and metadata for a given base folder + design letter.
    Folder convention: {base_folder}_design_{X}/
    Returns (states: np.ndarray float32, meta: pd.DataFrame) or (None, None) if not found.
    """
    folder = REPR_ROOT / f"{base_folder}_design_{design_letter}"
    states_path = folder / "hidden_states.npy"
    meta_path   = folder / "metadata.parquet"
    if not states_path.exists():
        return None, None
    states = np.load(str(states_path)).astype(np.float32)
    meta   = pd.read_parquet(meta_path)
    return states, meta


def load_combined(design_letter):
    """
    Load and combine harmful + benign states for a given design.
    Returns (states, meta) or (None, None) if either folder is missing.
    """
    sh, mh = load_design(HARMFUL_FOLDER, design_letter)
    sb, mb = load_design(BENIGN_FOLDER,  design_letter)
    if sh is None or sb is None:
        return None, None
    states = np.concatenate([sh, sb], axis=0)
    meta   = pd.concat([mh, mb], ignore_index=True)
    return states, meta


# Check which designs are available
print("Available designs:")
for d in DESIGNS:
    sh, mh = load_design(HARMFUL_FOLDER, d)
    sb, mb = load_design(BENIGN_FOLDER,  d)
    h_str = f"{len(mh)} rows" if mh is not None else "MISSING"
    b_str = f"{len(mb)} rows" if mb is not None else "MISSING"
    print(f"  Design {d}: harmful={h_str}, benign={b_str}")

## Task 1 — Harmful vs Benign

Train on pairs 0–79, test on 80–99. Label: 1 = harmful, 0 = benign.

In [ ]:
# ── Run Task 1 for all designs ────────────────────────────────────────────────
#
# For designs A/C/D/E: iterate over turn_idx values
# For design B:        iterate over k (context depth)
# For design F:        single point (final turn only)
#
# All results stored in task1_results dict: {design_letter: pd.DataFrame}

task1_results = {}

for design in DESIGNS:
    states, meta = load_combined(design)
    if states is None:
        print(f"Design {design}: skipped (data not found)")
        continue

    # x-axis column: Design B uses 'k', others use 'turn_idx'
    x_col = "k" if design == "B" else "turn_idx"

    rows = []
    for x_val in sorted(meta[x_col].unique()):
        m   = meta[x_col] == x_val
        X_k = states[m]
        y_k = meta.loc[m, "label"].values
        pid = meta.loc[m, "pair_id"].values

        train = pid <= TRAIN_MAX_PAIR_ID
        test  = pid >= TEST_MIN_PAIR_ID
        X_tr, y_tr = X_k[train], y_k[train]
        X_te, y_te = X_k[test],  y_k[test]

        if (y_tr == 0).sum() < MIN_EXAMPLES or (y_tr == 1).sum() < MIN_EXAMPLES:
            print(f"  Design {design} {x_col}={x_val}: skipped (low train coverage)")
            continue
        if (y_te == 0).sum() < MIN_TEST or (y_te == 1).sum() < MIN_TEST:
            print(f"  Design {design} {x_col}={x_val}: skipped (low test coverage)")
            continue

        res = probe_all(X_tr, y_tr, X_te, y_te)
        res[x_col] = x_val
        rows.append(res)
        print(f"  Design {design} {x_col}={x_val}: "
              f"LR={res['lr_auc']:.3f}  LAT={res['lat_auc']:.3f}  MLP={res['mlp_auc']:.3f}  "
              f"(train={res['n_train']}, test={res['n_test']})")

    if rows:
        task1_results[design] = pd.DataFrame(rows)
        print()

print("Task 1 done.")

In [ ]:
# ── Plot Task 1 — AUC curves per design ──────────────────────────────────────
COLORS = {"LR": "steelblue", "LAT": "darkorange", "MLP": "seagreen"}
METHOD_COLS = {"LR": "lr_auc", "LAT": "lat_auc", "MLP": "mlp_auc"}

available_designs = [d for d in DESIGNS if d in task1_results]
n_plots = len(available_designs)

fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 4), sharey=True)
if n_plots == 1:
    axes = [axes]

for ax, design in zip(axes, available_designs):
    df = task1_results[design]
    x_col = "k" if design == "B" else "turn_idx"
    x_label = "Context depth k" if design == "B" else "Turn"

    for method, col in METHOD_COLS.items():
        ax.plot(df[x_col], df[col], marker="o", label=method, color=COLORS[method])

    ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="Chance")
    ax.set_title(f"Design {design}", fontsize=12)
    ax.set_xlabel(x_label)
    ax.set_ylim(0.45, 1.02)
    ax.set_xticks(sorted(df[x_col].unique()))
    if ax is axes[0]:
        ax.set_ylabel("AUC (harmful vs benign)")
    ax.legend(fontsize=8)

fig.suptitle(f"{FRAMEWORK} — Task 1: Harmful vs Benign — AUC by Design", fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "07_task1_auc_by_design.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {FIG_DIR / '07_task1_auc_by_design.png'}")

## Task 2 — Outcome Probe (Jailbroken vs Refusal, within Harmful)

Trains within harmful conversations only. `near_miss` excluded as ambiguous.  
Label: 1 = jailbroken, 0 = refusal. Same pair-id train/test split.

In [ ]:
# ── Run Task 2 for all designs ────────────────────────────────────────────────
task2_results = {}

for design in DESIGNS:
    sh, mh = load_design(HARMFUL_FOLDER, design)
    if sh is None:
        print(f"Design {design}: skipped (harmful data not found)")
        continue

    # Filter to jailbroken / refusal only
    keep = mh["verdict"].isin(["jailbroken", "refusal"])
    meta_h = mh[keep].copy()
    meta_h["outcome"] = (meta_h["verdict"] == "jailbroken").astype(int)
    states_h = sh[keep.values]

    if len(meta_h) == 0:
        print(f"Design {design}: skipped (no jailbroken/refusal rows)")
        continue

    x_col = "k" if design == "B" else "turn_idx"

    rows = []
    for x_val in sorted(meta_h[x_col].unique()):
        m   = meta_h[x_col] == x_val
        idx = meta_h.index[m]  # index into original meta_h (before reindex)

        # Re-map indices to states_h positions
        # meta_h is a filtered view; its positional indices map to states_h rows
        pos = np.where(keep.values)[0]  # positions in original sh that survived filter
        # need local positions in states_h
        local_idx = np.where(meta_h[x_col].values == x_val)[0]

        X_k = states_h[local_idx]
        y_k = meta_h.iloc[local_idx]["outcome"].values
        pid = meta_h.iloc[local_idx]["pair_id"].values

        train = pid <= TRAIN_MAX_PAIR_ID
        test  = pid >= TEST_MIN_PAIR_ID
        X_tr, y_tr = X_k[train], y_k[train]
        X_te, y_te = X_k[test],  y_k[test]

        if (y_tr == 0).sum() < MIN_EXAMPLES or (y_tr == 1).sum() < MIN_EXAMPLES:
            print(f"  Design {design} {x_col}={x_val}: skipped "
                  f"(jb={( y_tr==1).sum()} ref={(y_tr==0).sum()} in train)")
            continue
        if (y_te == 0).sum() < MIN_TEST or (y_te == 1).sum() < MIN_TEST:
            print(f"  Design {design} {x_col}={x_val}: skipped (low test coverage)")
            continue

        res = probe_all(X_tr, y_tr, X_te, y_te)
        res[x_col] = x_val
        rows.append(res)
        print(f"  Design {design} {x_col}={x_val}: "
              f"LR={res['lr_auc']:.3f}  LAT={res['lat_auc']:.3f}  MLP={res['mlp_auc']:.3f}  "
              f"(train jb={(y_tr==1).sum()} ref={(y_tr==0).sum()}, test n={len(y_te)})")

    if rows:
        task2_results[design] = pd.DataFrame(rows)
        print()

print("Task 2 done.")

In [ ]:
# ── Plot Task 2 — AUC curves per design ──────────────────────────────────────
available_designs_t2 = [d for d in DESIGNS if d in task2_results]
n_plots = len(available_designs_t2)

fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 4), sharey=True)
if n_plots == 1:
    axes = [axes]

for ax, design in zip(axes, available_designs_t2):
    df = task2_results[design]
    x_col = "k" if design == "B" else "turn_idx"
    x_label = "Context depth k" if design == "B" else "Turn"

    for method, col in METHOD_COLS.items():
        ax.plot(df[x_col], df[col], marker="o", label=method, color=COLORS[method])

    ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="Chance")
    ax.set_title(f"Design {design}", fontsize=12)
    ax.set_xlabel(x_label)
    ax.set_ylim(0.45, 1.02)
    ax.set_xticks(sorted(df[x_col].unique()))
    if ax is axes[0]:
        ax.set_ylabel("AUC (jailbroken vs refusal)")
    ax.legend(fontsize=8)

fig.suptitle(f"{FRAMEWORK} — Task 2: Outcome Probe — AUC by Design", fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "07_task2_auc_by_design.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {FIG_DIR / '07_task2_auc_by_design.png'}")

## Summary: Cross-Design Comparison

Which design × method combination gives the highest AUC?  
Left: Task 1 (harmful vs benign). Right: Task 2 (outcome probe).

In [ ]:
# ── Summary tables: peak AUC per design × method ─────────────────────────────

def peak_auc_table(results_dict, task_name):
    """Return a DataFrame with peak AUC per design × method."""
    rows = []
    for design, df in results_dict.items():
        row = {"Design": design}
        for method, col in METHOD_COLS.items():
            row[f"{method}_peak_AUC"] = df[col].max()
            x_col = "k" if design == "B" else "turn_idx"
            row[f"{method}_peak_{x_col}"] = df.loc[df[col].idxmax(), x_col]
        rows.append(row)
    df_peak = pd.DataFrame(rows).set_index("Design")
    print(f"\n=== {task_name} — Peak AUC per design × method ===")
    display(df_peak.round(3))
    return df_peak

peak1 = peak_auc_table(task1_results, "Task 1: Harmful vs Benign")
peak2 = peak_auc_table(task2_results, "Task 2: Outcome Probe")

In [ ]:
# ── Cross-design AUC comparison plot ─────────────────────────────────────────
# For designs A/C/D/E/F: plot AUC vs turn for all designs on one axes per task.
# Design B gets its own panel (x-axis = k, not turn_idx).

turn_designs = [d for d in DESIGNS if d != "B"]
DESIGN_COLORS = {"A": "#1f77b4", "C": "#ff7f0e", "D": "#2ca02c",
                 "E": "#d62728", "F": "#9467bd"}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# ── Row 0: Task 1, all methods ────────────────────────────────────────────────
for col_idx, (method, m_col) in enumerate(METHOD_COLS.items()):
    ax = axes[0, col_idx]
    for design in turn_designs:
        if design not in task1_results:
            continue
        df = task1_results[design]
        x_col = "turn_idx"
        ls = "-" if design != "F" else "--"
        ax.plot(df[x_col], df[m_col], marker="o", linestyle=ls,
                color=DESIGN_COLORS.get(design, "gray"), label=f"Design {design}")
    # Design B on same axes with different marker
    if "B" in task1_results:
        df_b = task1_results["B"]
        ax.plot(df_b["k"], df_b[m_col], marker="s", linestyle=":",
                color="black", label="Design B (k)")
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.7)
    ax.set_title(f"Task 1 — {method}", fontsize=11)
    ax.set_xlabel("Turn / k")
    ax.set_ylim(0.45, 1.02)
    if col_idx == 0:
        ax.set_ylabel("AUC (harmful vs benign)")
    ax.legend(fontsize=7)

# ── Row 1: Task 2, all methods ────────────────────────────────────────────────
for col_idx, (method, m_col) in enumerate(METHOD_COLS.items()):
    ax = axes[1, col_idx]
    for design in turn_designs:
        if design not in task2_results:
            continue
        df = task2_results[design]
        x_col = "turn_idx"
        ls = "-" if design != "F" else "--"
        ax.plot(df[x_col], df[m_col], marker="o", linestyle=ls,
                color=DESIGN_COLORS.get(design, "gray"), label=f"Design {design}")
    if "B" in task2_results:
        df_b = task2_results["B"]
        ax.plot(df_b["k"], df_b[m_col], marker="s", linestyle=":",
                color="black", label="Design B (k)")
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.7)
    ax.set_title(f"Task 2 — {method}", fontsize=11)
    ax.set_xlabel("Turn / k")
    ax.set_ylim(0.45, 1.02)
    if col_idx == 0:
        ax.set_ylabel("AUC (jailbroken vs refusal)")
    ax.legend(fontsize=7)

fig.suptitle(f"{FRAMEWORK} — All Designs × All Methods", fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "07_summary_all_designs_methods.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {FIG_DIR / '07_summary_all_designs_methods.png'}")

In [ ]:
# ── Heatmap: peak AUC per design × method for both tasks ─────────────────────
import matplotlib.colors as mcolors

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, peak_df, title in zip(
    axes,
    [peak1, peak2],
    ["Task 1: Harmful vs Benign", "Task 2: Outcome (jb vs ref)"],
):
    # Extract only the AUC columns
    auc_cols = [c for c in peak_df.columns if c.endswith("_peak_AUC")]
    col_labels = [c.replace("_peak_AUC", "") for c in auc_cols]
    data = peak_df[auc_cols].values.astype(float)

    im = ax.imshow(data, vmin=0.5, vmax=1.0, cmap="RdYlGn", aspect="auto")
    ax.set_xticks(range(len(col_labels)))
    ax.set_xticklabels(col_labels, fontsize=10)
    ax.set_yticks(range(len(peak_df.index)))
    ax.set_yticklabels(peak_df.index, fontsize=10)
    ax.set_title(title, fontsize=11)

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data[i, j]
            color = "white" if val > 0.85 or val < 0.58 else "black"
            ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                    fontsize=9, color=color)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Peak AUC")

fig.suptitle(f"{FRAMEWORK} — Peak AUC Heatmap: Design × Method", fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "07_peak_auc_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {FIG_DIR / '07_peak_auc_heatmap.png'}")